# 🤖 Módulo 1 · Cuaderno 2 — Nexo aprende a pensar
### Técnicas de prompting, salidas estructuradas y evaluación sistemática

**AVE Ingeniería de Prompting y Agentes Autónomos · Universidad Ean**

Nexo ya responde con datos oficiales. Ahora la cooperativa necesita que **clasifique** los mensajes de los asociados, **calcule** liquidaciones sin equivocarse y responda siempre con el mismo estilo. Y, sobre todo, necesitamos **demostrar** que funciona bien, no solo creerlo.

**Al terminar podrás:**
1. Comparar *zero-shot* y *few-shot* (enseñar con ejemplos).
2. Usar *chain-of-thought* (razonar paso a paso).
3. Crear un **system prompt** y **plantillas reutilizables**.
4. Pedir respuestas con **estructura fija** (JSON) usando Pydantic.
5. Aplicar dos **prompt patterns**.
6. **Evaluar** a Nexo con un conjunto de pruebas: exactitud, consistencia y alucinaciones.

> Relación con el AVE: Módulo 1, Págs. 06 a 10.

## 0. Antes de empezar: cómo usar este cuaderno

Un **cuaderno de Colab** mezcla texto (como este) y **celdas de código** que puedes ejecutar.

- Para ejecutar una celda, haz clic en ella y presiona **Shift + Enter** (o el botón ▶ de la izquierda).
- Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla, vuelve a ejecutar desde la sección de configuración.
- Lo que escribe el código aparece debajo de la celda: eso es la **salida**.
- Si cierras la pestaña o pasa mucho tiempo, Colab reinicia la sesión y hay que ejecutar todo otra vez (*Entorno de ejecución → Ejecutar todas*).

### Tu clave de Gemini (gratis)

Nexo usa los modelos **Gemini** de Google a través de su API. Necesitas una clave personal:

1. Entra a <https://aistudio.google.com/apikey> con tu cuenta de Google y haz clic en **Create API key**.
2. Copia la clave.
3. En Colab, abre el panel izquierdo y haz clic en el ícono de la **llave 🔑 (Secretos)**.
4. Agrega un secreto con nombre `GOOGLE_API_KEY`, pega la clave en *Valor* y activa **Acceso al cuaderno**.

> ⚠️ **Nunca escribas tu clave dentro del código** ni la compartas: si entregas o compartes el cuaderno, cualquiera podría usarla. Los *Secretos* de Colab la guardan aparte.

La capa gratuita de Gemini tiene un límite de solicitudes por minuto. Si ves un error **429**, espera un minuto: el código de este cuaderno ya reintenta solo.

## 1. Configuración

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────
# El signo "!" ejecuta un comando de la terminal dentro de Colab.
# google-genai : la librería oficial de Google para usar los modelos Gemini.
# pydantic     : sirve para definir "moldes" de datos y pedir respuestas con estructura fija.
# -q (quiet)   : instala sin llenar la pantalla de mensajes.
!pip install -q -U google-genai pydantic

In [ ]:
# ── Configuración: conectar con Gemini ───────────────────────────────────
import time                      # para medir tiempos y esperar entre reintentos
import json                      # para leer y escribir datos en formato JSON
from google import genai         # cliente para hablar con los modelos Gemini
from google.genai import types   # "tipos": objetos de configuración (temperatura, herramientas, etc.)


def obtener_clave():
    """Lee la clave desde los Secretos de Colab; si no existe, la pide por teclado."""
    try:
        from google.colab import userdata
        clave = userdata.get("GOOGLE_API_KEY")
        if clave:
            return clave
    except Exception:
        pass  # no estamos en Colab o el secreto no tiene acceso al cuaderno
    from getpass import getpass  # getpass oculta lo que escribes
    return getpass("Pega tu clave de Gemini y presiona Enter: ")


# El "cliente" es el objeto que envía nuestras preguntas al modelo y trae las respuestas.
client = genai.Client(api_key=obtener_clave())

# Los nombres de los modelos cambian con el tiempo. Probamos una lista en orden de preferencia
# y nos quedamos con el primero que esté disponible para tu clave.
MODELOS_PREFERIDOS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash"]


def elegir_modelo(preferidos=MODELOS_PREFERIDOS):
    disponibles = []
    for m in client.models.list():
        acciones = m.supported_actions or []
        if "generateContent" in acciones:
            disponibles.append(m.name.replace("models/", ""))
    for nombre in preferidos:
        if nombre in disponibles:
            return nombre
    # Si ninguno de la lista existe, usamos cualquier modelo "flash" disponible.
    flash = [n for n in disponibles if "flash" in n and "image" not in n and "tts" not in n]
    return flash[0] if flash else disponibles[0]


MODELO = elegir_modelo()
print("✅ Conectado. Modelo que usará Nexo:", MODELO)


def llamar_con_reintentos(funcion, *args, intentos=5, **kwargs):
    """Ejecuta una llamada a la API y, si el servicio está saturado (error 429 o 503),
    espera y vuelve a intentar. Así el cuaderno no se detiene por límites de la capa gratuita."""
    espera = 10
    for intento in range(1, intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except Exception as error:
            mensaje = str(error)
            if any(codigo in mensaje for codigo in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and intento < intentos:
                print(f"⏳ Servicio ocupado (intento {intento}). Espero {espera} s y reintento...")
                time.sleep(espera)
                espera *= 2  # cada vez esperamos el doble
            else:
                raise


def preguntar(prompt, sistema=None, temperatura=None, modelo=None, **otros):
    """Envía un texto al modelo y devuelve la respuesta como texto.
    - prompt:      lo que le pedimos al modelo.
    - sistema:     instrucciones de sistema (quién es y cómo se comporta Nexo).
    - temperatura: 0 = respuestas más estables; 1 o más = respuestas más variadas.
    - otros:       cualquier otro ajuste de GenerateContentConfig."""
    config = types.GenerateContentConfig(system_instruction=sistema, temperature=temperatura, **otros)
    respuesta = llamar_con_reintentos(
        client.models.generate_content, model=modelo or MODELO, contents=prompt, config=config
    )
    return respuesta.text


print(preguntar("Saluda en una sola frase, en español."))

In [ ]:
# ── Datos (ficticios) de la Cooperativa Raíces ───────────────────────────
# Son la "fuente de verdad" de la cooperativa. Nexo no debe inventar nada que no esté aquí.
COOPERATIVA = {
    "nombre": "Cooperativa Raíces",
    "producto": "café pergamino seco",
    "precio_referencia_carga": 2_150_000,   # pesos colombianos por carga de 125 kg (se actualiza cada lunes)
    "fecha_precio": "lunes de esta semana",
    "horario_acopio": "lunes a sábado, de 6:00 a. m. a 2:00 p. m.",
}

REGLAMENTO = [
    "Art. 1. Precio de referencia: la cooperativa publica cada lunes el precio de referencia por carga de 125 kg de café pergamino seco. Nexo solo informa el precio vigente publicado.",
    "Art. 2. Calidad: el café se recibe con humedad entre 10 % y 12 %. Si la humedad supera el 12 %, se descuenta el 1 % del valor por cada punto adicional.",
    "Art. 3. Horario del centro de acopio: lunes a sábado de 6:00 a. m. a 2:00 p. m. Los domingos y festivos no hay recepción.",
    "Art. 4. Recogida en camión: solo para entregas de 10 cargas o más, en veredas con vía carreteable, y se solicita con 48 horas de anticipación.",
    "Art. 5. Pagos: los pagos se hacen por transferencia a la cuenta registrada del asociado dentro de los 3 días hábiles siguientes a la entrega.",
    "Art. 6. Aprobación de pagos: todo pago igual o superior a 5.000.000 de pesos requiere la aprobación de la analista de pagos.",
    "Art. 7. Anticipos de cosecha: un asociado puede pedir un anticipo de hasta el 30 % del valor entregado el año anterior.",
    "Art. 8. Datos personales: Nexo no comparte datos de un asociado con otra persona y nunca pide contraseñas ni claves bancarias.",
    "Art. 9. Quejas: toda queja se registra con un número de caso y la responde una persona del equipo en máximo 5 días hábiles.",
    "Art. 10. Asamblea: la asamblea general de asociados se realiza el último sábado de marzo.",
]

ASOCIADOS = {
    "A-101": {"nombre": "Aurelio Gómez", "vereda": "La Esperanza", "via_carreteable": True,  "cargas_anio_anterior": 48},
    "A-102": {"nombre": "Rosa Pineda",   "vereda": "El Diviso",    "via_carreteable": False, "cargas_anio_anterior": 22},
    "A-103": {"nombre": "Julián Ortiz",  "vereda": "Alto Bonito",  "via_carreteable": True,  "cargas_anio_anterior": 9},
}

print("Precio de referencia:", f"${COOPERATIVA['precio_referencia_carga']:,}".replace(",", "."), "por carga")
print("Artículos del reglamento:", len(REGLAMENTO), "| Asociados registrados:", len(ASOCIADOS))

## 2. Zero-shot vs. few-shot

Nexo recibe muchos mensajes y debe clasificarlos en una de estas categorías: `precio`, `recogida`, `pago`, `queja`, `otro`.

- **Zero-shot:** le damos la instrucción **sin ejemplos**.
- **Few-shot:** le damos la instrucción **con algunos ejemplos resueltos**. El modelo imita el patrón.

In [ ]:
CATEGORIAS = ["precio", "recogida", "pago", "queja", "otro"]

def clasificar_zero_shot(mensaje):
    prompt = f"""Clasifica el mensaje de un asociado de una cooperativa cafetera en UNA de estas categorías:
{", ".join(CATEGORIAS)}.
Responde solo con la categoría, en minúsculas.

Mensaje: {mensaje}"""
    return preguntar(prompt, temperatura=0).strip().lower()


# Los ejemplos se eligen para cubrir casos difíciles (mensajes ambiguos o con dos temas).
EJEMPLOS_FEW_SHOT = """Mensaje: ¿A cómo está la carga hoy?
Categoría: precio

Mensaje: El camión no pasó y se me mojó el café, estoy muy molesto
Categoría: queja

Mensaje: Ya entregué hace una semana y no veo la plata en la cuenta
Categoría: pago

Mensaje: ¿Me pueden mandar el carro por las 12 cargas?
Categoría: recogida

Mensaje: ¿Dónde queda la oficina?
Categoría: otro"""

def clasificar_few_shot(mensaje):
    prompt = f"""Clasifica el mensaje de un asociado de una cooperativa cafetera en UNA de estas categorías:
{", ".join(CATEGORIAS)}.
Responde solo con la categoría, en minúsculas.

{EJEMPLOS_FEW_SHOT}

Mensaje: {mensaje}
Categoría:"""
    return preguntar(prompt, temperatura=0).strip().lower()


mensaje = "Llevo tres días esperando la transferencia y nadie me contesta, qué falta de seriedad"
print("Zero-shot:", clasificar_zero_shot(mensaje))
print("Few-shot: ", clasificar_few_shot(mensaje))

Este mensaje es ambiguo: habla de un **pago**, pero el tono es de **queja**. Los ejemplos le dicen al modelo **qué criterio** usa la cooperativa (en el ejemplo, un problema de servicio con molestia se clasifica como queja). Los ejemplos enseñan el formato y el criterio, no solo el tema (Min et al., 2022).

## 3. Chain-of-thought: pensar paso a paso

Para tareas con varios pasos (por ejemplo, calcular cuánto se le paga a un asociado), pedirle al modelo que **muestre su razonamiento** antes de la respuesta final reduce errores.

Caso: Don Aurelio entrega 14 cargas con humedad del 13,5 %. Según el Art. 2, se descuenta 1 % por cada punto de humedad por encima de 12 %.

In [ ]:
caso = f"""Precio de referencia: {COOPERATIVA['precio_referencia_carga']} pesos por carga.
Regla de calidad: {REGLAMENTO[1]}
Don Aurelio entrega 14 cargas con humedad de 13,5 %. ¿Cuánto se le debe pagar?"""

directo = preguntar(caso + "\nResponde solo con el valor final en pesos.", temperatura=0)

paso_a_paso = preguntar(
    caso + """
Razona paso a paso:
1. Calcula el valor bruto (cargas × precio).
2. Calcula los puntos de humedad por encima de 12 %.
3. Calcula el porcentaje y el valor del descuento.
4. Calcula el valor final.
Al final escribe una línea: VALOR FINAL: <número>""",
    temperatura=0,
)
print("🔹 Directo:\n", directo)
print("\n🔹 Paso a paso:\n", paso_a_paso)

Comprobemos con Python cuál es el valor correcto. **Nunca confíes en un cálculo del modelo sin verificarlo:**

In [ ]:
cargas, humedad = 14, 13.5
bruto = cargas * COOPERATIVA["precio_referencia_carga"]
descuento_pct = max(0, humedad - 12) * 1      # 1 % por punto sobre 12 %
valor_final = bruto * (1 - descuento_pct / 100)
print(f"Bruto: {bruto:,.0f} | Descuento: {descuento_pct} % | Valor final: {valor_final:,.0f}".replace(",", "."))

> 💡 **Lección de diseño:** para cálculos exactos, lo mejor es que el modelo **use una función de Python** en lugar de calcular "de cabeza". Eso es exactamente lo que haremos en el Módulo 2 con *function calling*.

## 4. System prompt y plantillas reutilizables

El **system prompt** son las instrucciones permanentes de Nexo: quién es, qué puede hacer y qué no. Va aparte de la pregunta del usuario y se aplica a toda la conversación.

Una **plantilla** es un prompt con espacios en blanco (`$variable`) que se llenan en cada uso. Así todos los mensajes de Nexo siguen el mismo estándar.

In [ ]:
from string import Template   # Template permite plantillas con $variables

SYSTEM_NEXO = f"""Eres Nexo, el asistente virtual de la {COOPERATIVA['nombre']}.
OBJETIVO: resolver consultas de los asociados sobre precios, recogidas, pagos y reglamento.
REGLAS:
- Usa solo la información del reglamento y de los datos oficiales que se te entreguen.
- Si no tienes el dato, dilo y ofrece comunicar al asociado con una persona.
- Nunca pidas contraseñas ni datos bancarios (Art. 8).
- Responde en español sencillo, con trato de "usted", máximo 4 frases.
- Cita el artículo del reglamento que respalda tu respuesta.
DATOS OFICIALES:
Precio de referencia: {COOPERATIVA['precio_referencia_carga']} pesos por carga ({COOPERATIVA['fecha_precio']}).
REGLAMENTO:
""" + "\n".join(REGLAMENTO)

PLANTILLA_CONSULTA = Template("""Asociado: $nombre (vereda $vereda).
Mensaje: $mensaje""")

consulta = PLANTILLA_CONSULTA.substitute(
    nombre="Rosa Pineda", vereda="El Diviso",
    mensaje="¿Puedo pedir un adelanto para pagar los recolectores?"
)
print(preguntar(consulta, sistema=SYSTEM_NEXO, temperatura=0.3))

### Conversaciones con memoria de corto plazo
Con `client.chats.create` el historial se guarda solo y Nexo recuerda lo que se dijo antes en la misma conversación:

In [ ]:
chat = client.chats.create(model=MODELO, config=types.GenerateContentConfig(system_instruction=SYSTEM_NEXO, temperature=0.3))

for mensaje in ["Hola, soy Julián Ortiz, de Alto Bonito.",
                "¿Cuánto me darían por 9 cargas?",
                "¿Y me las recogen en camión?"]:
    respuesta = llamar_con_reintentos(chat.send_message, mensaje)
    print(f"👤 {mensaje}\n🤖 {respuesta.text.strip()}\n")

## 5. Salidas estructuradas (JSON)

Para que otro programa use lo que dice Nexo (por ejemplo, guardar una solicitud), la respuesta debe tener **siempre los mismos campos**. Con **Pydantic** definimos el "molde" y Gemini responde respetándolo.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class Solicitud(BaseModel):
    """Molde de una solicitud de un asociado."""
    categoria: Literal["precio", "recogida", "pago", "queja", "otro"]
    resumen: str = Field(description="Resumen de la solicitud en una frase")
    cargas: Optional[int] = Field(description="Número de cargas mencionadas, si las hay")
    urgente: bool = Field(description="True si el asociado expresa urgencia o molestia")

def extraer_solicitud(mensaje):
    config = types.GenerateContentConfig(
        temperature=0,
        response_mime_type="application/json",   # pedimos JSON...
        response_schema=Solicitud,               # ...con este molde exacto
    )
    r = llamar_con_reintentos(client.models.generate_content, model=MODELO,
                              contents=f"Extrae la solicitud de este mensaje:\n{mensaje}", config=config)
    return r.parsed or Solicitud.model_validate_json(r.text)   # r.parsed ya es un objeto Solicitud

s = extraer_solicitud("¡Urgente! Tengo 15 cargas listas en La Esperanza y se me va a mojar, manden el camión")
print(s)
print("Categoría:", s.categoria, "| Cargas:", s.cargas, "| Urgente:", s.urgente)

## 6. Prompt patterns

Un **prompt pattern** es una solución reutilizable a un problema frecuente al diseñar prompts (White et al., 2023). Probemos dos:

- **Interacción invertida:** el modelo pregunta lo que le falta antes de responder.
- **Verificador:** el modelo revisa su propia respuesta contra el reglamento.

In [ ]:
# Patrón 1: interacción invertida
patron_invertido = SYSTEM_NEXO + """
PATRÓN: si a la consulta le falta información necesaria (número de cargas, vereda o código de asociado),
NO respondas todavía: haz UNA pregunta para obtener el dato que falta."""
print(preguntar("Quiero que me recojan el café", sistema=patron_invertido, temperatura=0))

In [ ]:
# Patrón 2: verificador. Primero una respuesta "borrador" y luego una revisión contra el reglamento.
borrador = "Claro, le enviamos el camión mañana por sus 6 cargas sin ningún problema."
revision = preguntar(f"""Eres el verificador de calidad de Nexo.
Revisa si la RESPUESTA cumple el REGLAMENTO. Indica: CUMPLE o NO CUMPLE, el artículo afectado
y una versión corregida de la respuesta.

REGLAMENTO:
{chr(10).join(REGLAMENTO)}

RESPUESTA: {borrador}""", temperatura=0)
print(revision)

## 7. Evaluación sistemática: ¿Nexo funciona o solo lo parece?

Probar "a ojo" con dos o tres preguntas no basta. Vamos a construir un **conjunto de pruebas** con respuestas esperadas y medir:

1. **Exactitud:** ¿cuántas clasificaciones acierta?
2. **Consistencia:** ¿responde lo mismo si repetimos la pregunta?
3. **Alucinación:** ¿inventa cifras que no están en los datos?

In [ ]:
PRUEBAS = [
    ("¿A cómo está el café hoy?", "precio"),
    ("Mándenme el camión, tengo 20 cargas", "recogida"),
    ("No me ha llegado el pago de la semana pasada", "pago"),
    ("El funcionario me trató muy mal, quiero poner una queja", "queja"),
    ("¿Cuándo es la asamblea?", "otro"),
    ("¿Cuánto pagan por carga y cuándo consignan?", "precio"),
    ("Exijo que me respondan ya, el camión me dejó plantado", "queja"),
    ("¿Puedo pedir un anticipo?", "pago"),
]

filas = []
for mensaje, esperado in PRUEBAS:
    zs, fs = clasificar_zero_shot(mensaje), clasificar_few_shot(mensaje)
    filas.append({"mensaje": mensaje, "esperado": esperado, "zero_shot": zs, "few_shot": fs})

import pandas as pd
resultados = pd.DataFrame(filas)
resultados["acierto_zs"] = resultados["zero_shot"] == resultados["esperado"]
resultados["acierto_fs"] = resultados["few_shot"] == resultados["esperado"]
print(f"Exactitud zero-shot: {resultados['acierto_zs'].mean():.0%}")
print(f"Exactitud few-shot:  {resultados['acierto_fs'].mean():.0%}")
resultados

In [ ]:
# Consistencia: repetimos la misma pregunta 3 veces y contamos respuestas distintas.
import re

def cifras(texto):
    """Extrae los números de un texto (quitando puntos o comas de miles)."""
    return {int(re.sub(r"[.,]", "", n)) for n in re.findall(r"\d[\d.,]{3,}\d", texto)}

pregunta = "¿Cuál es el precio de referencia por carga?"
respuestas = [preguntar(pregunta, sistema=SYSTEM_NEXO, temperatura=1) for _ in range(3)]
for r in respuestas:
    print("•", r.strip())

# Alucinación: ¿aparece alguna cifra que no sea el precio oficial?
oficial = COOPERATIVA["precio_referencia_carga"]
inventadas = set().union(*[cifras(r) for r in respuestas]) - {oficial, 125}
print("\nCifras que no están en los datos oficiales:", inventadas or "ninguna ✅")

> **Para tu bitácora:** guarda la tabla de resultados. En tu proyecto final deberás presentar una tabla parecida con **mínimo cinco casos de prueba** (entrada, resultado esperado, resultado obtenido y ajuste).

## 8. Reto para tu proyecto 🎯

1. Escribe el **system prompt** de tu agente: objetivo, reglas, datos oficiales y límites.
2. Define una **salida estructurada** (clase Pydantic) para la información que tu agente debe registrar.
3. Crea un conjunto de **al menos 8 pruebas** con respuestas esperadas y mide la exactitud de tu agente.
4. Guarda al menos **dos versiones** de tu system prompt y anota qué cambiaste y por qué: lo pediremos en el documento final.

**Lecturas de apoyo:** Min et al. (2022); Chu et al. (2024); White et al. (2023); Reynolds y McDonell (2021).

## 📄 ¿Cómo guardar este cuaderno en PDF con todas las salidas?

1. *Entorno de ejecución → Reiniciar y ejecutar todas* (o *Ejecutar todas*). Espera a que todas las celdas terminen.
2. Revisa que cada celda muestre su salida y que no haya errores en rojo.
3. *Archivo → Imprimir* y, en *Destino*, elige **Guardar como PDF**.
4. Para el `.ipynb`: *Archivo → Descargar → Descargar .ipynb*.